# NewsBot Intelligence System 2.0 — Final executable notebook
José Miguel Tuozzo Izaguirre · ITAI 2373 · Houston Community College

This notebook runs the actual final library. The 30 records are fictional; historical BBC results are reported separately. Optional neural evidence is produced by the separate multilingual job.

In [ ]:
from pathlib import Path
import sys, json, os
root = Path.cwd()
if not (root / 'newsbot2').is_dir():
    root = next(p for p in (root/'final-newsbot-2.0',root.parent) if (p/'newsbot2').is_dir())
os.chdir(root)
sys.path.insert(0,str(root))
from newsbot2.io import import_articles
from newsbot2.engine import NewsBotEngine
from newsbot2.evaluation import evaluate_classification_cv
from IPython.display import display
import pandas as pd
articles = import_articles(root/'data/demo_news.jsonl')
engine = NewsBotEngine(articles)
assert len(articles)==30 and all(a.synthetic for a in articles)
display(engine.overview())

## 1. Advanced content analysis
Inspect topics, category fit, sentiment, entity candidates and provenance. Local fit is not independent validation.

In [ ]:
display(engine.train_local_classifier())
analysis=engine.analyze('demo-tech-1')
display({k:analysis[k] for k in ('classification','sentiment','entities','provenance')})
topics=engine.topics(5)
display(pd.DataFrame(topics['topics']))

## 2. Language understanding and generation
The summarizer selects source sentences; headline suggestions use templates. Compare sparse and latent semantic retrieval.

In [ ]:
display(analysis['summary'],analysis['enhancement'])
lsa=NewsBotEngine(articles,retrieval_mode='lsa')
for e in (engine,lsa):
    display(pd.DataFrame(e.search('robot cameras sensors navigation',k=3))[['article_id','score','retrieval_method']])

## 3. Multilingual intelligence
Default lexical pivot is limited; the separate neural test installs real weights and translation packs.

In [ ]:
from newsbot2.language import detect_language
for query in ['investigación de inteligencia artificial','technologie robotique','forschung daten']:
    display(query,detect_language(query),pd.DataFrame(engine.search(query,k=3)))
display(engine.search('robótica navegación universidad',k=3,language='es'))
display(engine.translate('La universidad investiga inteligencia artificial.','es','en'))

## 4. Conversation and abstention
Only source excerpts are returned. Relevance is not truth.

In [ ]:
answer=engine.ask('What did the AI chip researchers measure?')
display(answer)
assert answer['citations']
display(engine.ask('Tell me more'))
unknown=engine.ask('zzqxv nonexistent topic 0123456789')
display(unknown)
assert unknown['status']=='insufficient_evidence'

## 5. Experimental evaluation
Stratified out-of-fold prediction fits features inside training folds. Controlled synthetic benchmark only.

In [ ]:
cv=evaluate_classification_cv(articles,folds=3)
display(pd.DataFrame(cv['per_class']).T)
display(pd.DataFrame(cv['confusion_matrix'],index=cv['classes'],columns=cv['classes']))
import subprocess
subprocess.run([sys.executable,'run_research.py'],check=True)
research=json.loads((root/'artifacts/research_results.json').read_text())
display({k:research['retrieval_probes'][k] for k in ('n','k','pivot_mrr','naive_mrr','pivot_recall_at_k','naive_recall_at_k')})
display(research['paired_multilingual_extension'])

## 6. Rubric extensions: LDA, NMF, clusters, timelines and intent routing

In [ ]:
from newsbot2.advanced import discover_topics, sentiment_timeline, entity_network, language_comparison, structured_query
for method in ('lda','nmf'):
    result=discover_topics(articles,4,method)
    display(result['method'],pd.DataFrame(result['topics']),pd.DataFrame(result['evolution']))
display(sentiment_timeline(articles),language_comparison(articles))
display(pd.DataFrame(entity_network(articles)))
display(structured_query(engine,'Compare tech topics across languages'))

## 6. Engineering evidence
Automated checks, browser captures and neural evidence are maintained independently. Scientific limitations: selected fictional probes, no independent real multilingual benchmark, uncalibrated scores, no factual verification.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','tests','-q'],check=True)
print('Notebook completed from a fresh kernel.')